# Gym Environment Tutorial

This notebook provides a step-by-step guide to understanding the quantum circuit gymnasium environment. We'll start simple with a single circuit and gradually explore all the components.

1. **Creating a simple dataset** - Start with just one circuit to understand the basics
2. **Understanding observations** - How the agent "sees" the circuit (sliding window)
3. **Understanding actions** - How the agent applies noise channels
4. **Understanding rewards** - How we measure the quality of noise predictions
5. **Complete episodes** - Put it all together

The environment is a game where:
- **Observation**: A sliding window view of the circuit
- **Action**: Apply noise channels (depolarizing, damping, coherent errors) at each position
- **Reward**: How well your noisy circuit matches the real noisy behavior (computed at the end)

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qibo import Circuit, gates

from rlnoise.config import (
    DatasetConfig,
    NoiseConfig,
    GateSpecificNoise,
    GymEnvConfig,
    RewardConfig,
)
from rlnoise.dataset import DatasetGenerator, CircuitDataset
from rlnoise.circuit_encoder import CircuitEncoder
from rlnoise.gym_env import QuantumCircuitEnv

# Set random seed for reproducibility
np.random.seed(42)

## 2. Start Simple: Create a Dataset with ONE Circuit

Let's begin with just one circuit to understand the basics. We'll create a simple 1-qubit circuit with a few gates.

In [ ]:
# Step 1: Configure the dataset - just 1 circuit, 1 qubit, 5 moments
dataset_config = DatasetConfig(
    n_circuits=1,           # Start with just ONE circuit
    qubits=1,               # One qubit to keep it simple
    moments=5,              # 5 time steps (gates)
    primitive_gates=["rx", "rz"],  # Only RX and RZ gates
    clifford=True,          # Generate valid quantum circuits
    seed=42
)

# Step 2: Configure noise - depolarizing and damping on both gates
noise_config = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rx", noise_channel="damping", noise_parameter=0.02),
        GateSpecificNoise(gate="rz", noise_channel="damping", noise_parameter=0.02),
    ]
)

# Step 3: Generate the dataset
generator = DatasetGenerator(dataset_config, noise_config)
dataset = generator.generate(verbose=True)

print(f"\n{'='*50}")
print(f"Dataset created!")
print(f"{'='*50}")
print(f"Number of circuits: {len(dataset)}")
print(f"Circuit shape: {dataset.circuits.shape}")
print(f"  - {dataset.circuits.shape[0]} time steps (moments)")
print(f"  - {dataset.circuits.shape[1]} qubit(s)")
print(f"  - {dataset.circuits.shape[2]} encoding features")
print(f"\nLabel (target density matrix) shape: {dataset.labels.shape}")
print(f"  - {dataset.labels.shape[0]} circuits")
print(f"  - {dataset.labels.shape[1]}x{dataset.labels.shape[2]} density matrix")

## 3. Understanding the Circuit Encoding

Before we create the environment, let's understand how circuits are encoded. Each circuit is represented as a 3D array with encoded gate information.

In [ ]:
# Let's look at our single circuit
circuit_array = dataset.circuits[0]  # Shape: (moments, qubits, encoding_dim)

print(f"Our circuit encoding shape: {circuit_array.shape}")
print(f"\nWhat's in the encoding (per qubit, per moment):")
print(f"  Index 0: Gate type (one-hot encoded)")
print(f"  Index 1-3: Gate parameters (angles, etc.)")
print(f"  Index 4: Depolarizing noise")
print(f"  Index 5: Damping/reset noise")
print(f"  Index 6: Coherent Z error (epsilon_z)")
print(f"  Index 7: Coherent X error (epsilon_x)")

print(f"\nLet's examine moment 0, qubit 0:")
print(circuit_array[0, 0])
print(f"\nNotice: Indices 4-7 contain the noise parameters we're trying to learn!")

# Visualize the circuit structure
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot gates
ax = axes[0]
gate_info = circuit_array[:, 0, 0:4]  # First 4 indices contain gate info
im = ax.imshow(gate_info.T, aspect='auto', cmap='viridis')
ax.set_xlabel('Moment (time step)')
ax.set_ylabel('Gate encoding index')
ax.set_title('Gate Structure')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Gate type', 'Param 1', 'Param 2', 'Param 3'])
plt.colorbar(im, ax=ax)

# Plot noise parameters (what the agent will learn)
ax = axes[1]
noise_info = circuit_array[:, 0, 4:8]  # Indices 4-7 contain noise
im = ax.imshow(noise_info.T, aspect='auto', cmap='RdYlGn_r')
ax.set_xlabel('Moment (time step)')
ax.set_ylabel('Noise parameter index')
ax.set_title('Noise Parameters (Agent\'s Action Space)')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Depolarizing', 'Damping', 'Coherent Z', 'Coherent X'])
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 4. Create the Environment

Now let's create the gymnasium environment. We need:
1. The dataset (we have it)
2. A circuit encoder (to convert between arrays and circuits)
3. Environment configuration (how the agent interacts)
4. Reward configuration (how we measure success)

In [ ]:
# Step 1: Create circuit encoder
encoder = CircuitEncoder(
    primitive_gates=["rx", "rz"],
    qubits=1
)

# Step 2: Configure environment behavior
env_config = GymEnvConfig(
    kernel_size=3,                      # Sliding window size (must be odd)
    action_space_max_value=0.1,         # Maximum noise value
    val_split=0.0,                      # No validation split (only 1 circuit)
    enable_only_depolarizing=False,     # Allow all noise types
)

# Step 3: Configure reward function
reward_config = RewardConfig(
    metric="trace",                     # Use trace distance metric
    function="inverted_squared",        # Transform: 1/(alpha * distance^2)
    alpha=20.0,                         # Scaling factor
)

# Step 4: Create the environment
env = QuantumCircuitEnv(
    dataset=dataset,
    encoder=encoder,
    env_config=env_config,
    reward_config=reward_config,
    primitive_gates=["rx", "rz"],
)

print("Environment created!")
print(f"Observation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"  - {env.observation_space.shape[0]} encoding features")
print(f"  - {env.observation_space.shape[1]} qubit(s)")
print(f"  - {env.observation_space.shape[2]} window size (kernel)")
print(f"\nAction space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")
print(f"  - {env.action_space.shape[0]} qubit(s)")
print(f"  - {env.action_space.shape[1]} noise parameters per qubit")

## 5. How Observations Work: The Sliding Window

The agent doesn't see the whole circuit at once. Instead, it sees a "sliding window" of 3 moments (like looking through a small window as you move along the circuit).

### 5.1. Reset the Environment and Get Initial Observation

In [ ]:
# Reset environment to start an episode
obs, info = env.reset()

print(f"Initial observation shape: {obs.shape}")
print(f"  - {obs.shape[0]} encoding features")
print(f"  - {obs.shape[1]} qubit(s)")
print(f"  - {obs.shape[2]} window positions")
print(f"\nCurrent position: {info}")
print(f"\nThe observation is showing moments: [0, 1, 2] (kernel_size=3)")

### 5.2. Visualize the Sliding Window

In [ ]:
# The observation is a 3D tensor: (encoding_dim, n_qubits, kernel_size)
# For our case: (8, 1, 3)

print("Observation at position 0:")
print("Shape:", obs.shape)
print("\nFirst qubit's window (8 features x 3 moments):")
print(obs[:, 0, :])

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(obs[:, 0, :], aspect='auto', cmap='viridis')
plt.xlabel('Window position (moment)')
plt.ylabel('Encoding feature')
plt.title('Observation: Sliding Window at Position 0')
plt.colorbar()
plt.yticks(range(8), ['Gate', 'Param1', 'Param2', 'Param3', 
                       'Depol', 'Damp', 'Coh-Z', 'Coh-X'])
plt.xticks(range(3), ['Moment 0', 'Moment 1', 'Moment 2'])
plt.tight_layout()
plt.show()

print("\n💡 Key insight: The agent focuses on the MIDDLE moment (index 1)")
print("   That's where it will apply its action!")

## 6. How Actions Work: Applying Noise Parameters

The agent's action specifies noise parameters to apply at the current position (the middle of the sliding window).

In [ ]:
# Action space shape: (n_qubits, 4)
# For 1 qubit: (1, 4)
# 4 noise parameters per qubit:
#   [0] epsilon_x (coherent X error)
#   [1] epsilon_z (coherent Z error)
#   [2] reset/damping probability
#   [3] depolarizing lambda

print("Action space details:")
print(f"  Shape: {env.action_space.shape}")
print(f"  Range: [{env.action_space.low[0,0]:.1f}, {env.action_space.high[0,0]:.1f}]")
print(f"  Scaled by: {env_config.action_space_max_value}")
print(f"\n  Index 0: Coherent X error (epsilon_x)")
print(f"  Index 1: Coherent Z error (epsilon_z)")
print(f"  Index 2: Damping/reset probability")
print(f"  Index 3: Depolarizing lambda")

# Sample a random action
random_action = env.action_space.sample()
print(f"\nRandom action example:")
print(random_action)
print(f"\nThis means: Apply these noise values to qubit 0:")
print(f"  epsilon_x = {random_action[0,0]:.4f}")
print(f"  epsilon_z = {random_action[0,1]:.4f}")
print(f"  damping   = {random_action[0,2]:.4f}")
print(f"  depol     = {random_action[0,3]:.4f}")

### 6.1. Take an Action and See What Happens

In [ ]:
# Let's apply a specific action: small depolarizing noise
action = np.zeros((1, 4), dtype=np.float32)
action[0, 3] = 0.5  # Set depolarizing to 0.5 (will be scaled by action_max=0.1)

print("Action we're taking:")
print(action)
print(f"\nThis will add depolarizing noise of: 0.5 * {env_config.action_space_max_value} = {0.5 * env_config.action_space_max_value}")

# Take the action
obs_next, reward, terminated, truncated, info = env.step(action)

print(f"\n{'='*50}")
print(f"After taking action:")
print(f"{'='*50}")
print(f"Terminated: {terminated} (episode ends when we reach the end)")
print(f"Position: {env.position}")
print(f"Reward: {reward:.6f} (0 until terminal state)")
print(f"\nNew observation shape: {obs_next.shape}")
print(f"Now viewing moments: [1, 2, 3] (window moved forward)")

### 6.2. Visualize How Actions Modify the Circuit

In [ ]:
# Let's see how the action modified the circuit's noise parameters
# The agent modifies indices 4-7 (noise parameters) at the current position

print("Circuit noise parameters at position 0 (where we just acted):")
print(f"Depolarizing (index 4): {env.current_circuit[0, 0, 4]:.4f}")
print(f"Damping (index 5):      {env.current_circuit[0, 0, 5]:.4f}")
print(f"Coherent Z (index 6):   {env.current_circuit[0, 0, 6]:.4f}")
print(f"Coherent X (index 7):   {env.current_circuit[0, 0, 7]:.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Before (original circuit noise - all zeros since we haven't acted on later positions)
ax = axes[0]
noise_original = dataset.circuits[0][:, 0, 4:8].T
im = ax.imshow(noise_original, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.1)
ax.set_xlabel('Moment (position)')
ax.set_ylabel('Noise type')
ax.set_title('Original Circuit (Clean)')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Depol', 'Damp', 'Coh-Z', 'Coh-X'])
plt.colorbar(im, ax=ax)

# After action (with noise applied at position 0)
ax = axes[1]
noise_modified = env.current_circuit[:, 0, 4:8].T
im = ax.imshow(noise_modified, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.1)
ax.set_xlabel('Moment (position)')
ax.set_ylabel('Noise type')
ax.set_title('After Action at Position 0')
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(['Depol', 'Damp', 'Coh-Z', 'Coh-X'])
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Action applied here')
ax.legend()
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

print("\n✅ You can see the depolarizing noise (row 0) increased at position 0!")

## 7. Understanding Rewards: How We Measure Success

The reward is computed ONLY at the terminal state (end of episode). It measures how well the noisy circuit matches the target noisy behavior.

In [ ]:
from rlnoise.reward import RewardFunction

# The reward function has two components:
# 1. A METRIC: measures distance between density matrices
# 2. A TRANSFORM: converts distance to reward (higher is better)

print("Our reward configuration:")
print(f"  Metric: {reward_config.metric}")
print(f"  Function: {reward_config.function}")
print(f"  Alpha: {reward_config.alpha}")

print("\n📊 Available metrics:")
print("  - 'trace': Trace distance (0 to 1)")
print("  - 'fidelity': 1 - fidelity (0 to 1)")
print("  - 'mse': Mean squared error")
print("  - 'mae': Mean absolute error")

print("\n🔄 Available transforms:")
print("  - 'linear': -alpha * distance")
print("  - 'log': -log(alpha * distance)")
print("  - 'inverted': 1 / (alpha * distance)")
print("  - 'inverted_squared': 1 / (alpha * distance^2)")

print(f"\n💡 With our settings:")
print(f"   If distance = 0.01, reward = 1 / (20 * 0.01^2) = 1 / 0.002 = 500")
print(f"   If distance = 0.1,  reward = 1 / (20 * 0.1^2)  = 1 / 0.2   = 5")
print(f"   Smaller distance → BIGGER reward!")

### 7.1. Testing the Reward Function

In [ ]:
# Let's test the reward function with example density matrices
reward_fn = RewardFunction(reward_config)

# Create two example density matrices
# Perfect match (identical)
dm1 = np.array([[1, 0], [0, 0]], dtype=complex)  # Pure state |0⟩
dm2 = np.array([[1, 0], [0, 0]], dtype=complex)  # Same state

# Compute metrics
metrics_perfect = reward_fn.evaluate(dm1, dm2)

print("Perfect match (identical density matrices):")
for key, value in metrics_perfect.items():
    print(f"  {key}: {value:.6f}")

print("\n" + "="*50)

# Different states
dm3 = np.array([[0.5, 0], [0, 0.5]], dtype=complex)  # Mixed state

metrics_different = reward_fn.evaluate(dm1, dm3)

print("\nDifferent states (|0⟩ vs maximally mixed):")
for key, value in metrics_different.items():
    print(f"  {key}: {value:.6f}")

# Visualize reward vs distance
distances = np.linspace(0.001, 0.5, 100)
rewards = [1.0 / (reward_config.alpha * d**2) for d in distances]

plt.figure(figsize=(10, 5))
plt.plot(distances, rewards, linewidth=2)
plt.xlabel('Distance')
plt.ylabel('Reward')
plt.title(f'Reward Function: {reward_config.function} (alpha={reward_config.alpha})')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\n💡 Key insight: The reward grows exponentially as the distance approaches 0!")

## 8. Complete Episode: Putting It All Together

Now let's run a complete episode from start to finish. We'll step through the circuit, apply actions, and get the final reward.

In [ ]:
# Reset the environment
obs, info = env.reset()

print(f"Starting episode with circuit {info['circuit_idx']}")
print(f"Circuit length: {info['circuit_length']} moments")
print(f"Need to take {info['circuit_length']} actions\n")

# Store trajectory
observations = [obs]
actions_taken = []
rewards_received = []

# Run episode with random actions
terminated = False
step = 0

while not terminated:
    # Sample random action
    action = env.action_space.sample()
    
    # Take step
    obs, reward, terminated, truncated, info = env.step(action)
    
    # Store
    observations.append(obs)
    actions_taken.append(action)
    rewards_received.append(reward)
    
    print(f"Step {step}: Position {step}, Reward = {reward:.6f}, Terminated = {terminated}")
    step += 1

print(f"\n{'='*50}")
print(f"Episode finished!")
print(f"{'='*50}")
print(f"Total steps: {step}")
print(f"Final reward: {rewards_received[-1]:.6f}")
print(f"All intermediate rewards were 0.0 (reward only at end)")

# Visualize the actions taken
actions_array = np.array(actions_taken).squeeze()  # Shape: (n_steps, 4)

plt.figure(figsize=(12, 5))
plt.imshow(actions_array.T, aspect='auto', cmap='RdYlGn')
plt.xlabel('Step (moment)')
plt.ylabel('Action component')
plt.yticks([0, 1, 2, 3], ['epsilon_x', 'epsilon_z', 'damping', 'depol'])
plt.title('Actions Taken During Episode')
plt.colorbar(label='Action value [0, 1]')
plt.tight_layout()
plt.show()

### 8.1. Inspecting the Final Noisy Circuit

In [ ]:
# The environment stores the circuit with all noise applied
final_circuit_array = env.current_circuit

print("Final circuit encoding with noise:")
print("Shape:", final_circuit_array.shape)
print("\nNoise parameters (indices 4-7) at each moment:")
print("Moment | Depol  | Damp   | Coh-Z  | Coh-X")
print("-" * 50)
for i in range(final_circuit_array.shape[0]):
    noise = final_circuit_array[i, 0, 4:8]
    print(f"  {i}    | {noise[0]:.4f} | {noise[1]:.4f} | {noise[2]:.4f} | {noise[3]:.4f}")

## 9. Customization: Adapting the Environment

Now that you understand the basics, let's explore how to customize the environment for your specific needs.

In [ ]:
### 9.1. Creating a Multi-Circuit Dataset

For real training, you need multiple circuits. Let's create a proper dataset.


### 9.2. Larger Dataset with Train/Val Split

In [ ]:
# Create a larger dataset
large_dataset_config = DatasetConfig(
    n_circuits=100,           # 100 circuits
    qubits=1,
    moments=10,
    primitive_gates=["rx", "rz"],
    clifford=True,
    seed=42
)

large_generator = DatasetGenerator(large_dataset_config, noise_config)
large_dataset = large_generator.generate(verbose=False)

# Create environment with validation split
large_env_config = GymEnvConfig(
    kernel_size=3,
    action_space_max_value=0.1,
    val_split=0.2,              # 20% for validation
    enable_only_depolarizing=False,
)

large_env = QuantumCircuitEnv(
    dataset=large_dataset,
    encoder=encoder,
    env_config=large_env_config,
    reward_config=reward_config,
    primitive_gates=["rx", "rz"],
)

print(f"Large environment created!")
print(f"Total circuits: {large_env.n_circuits}")
print(f"Training circuits: {large_env.n_circuits_train}")
print(f"Validation circuits: {large_env.n_circuits - large_env.n_circuits_train}")

# Reset will randomly select a training circuit
obs, info = large_env.reset()
print(f"\nReset selected training circuit: {info['circuit_idx']}")

# To select a specific circuit (e.g., for validation)
val_circuit_idx = large_env.n_circuits_train  # First validation circuit
obs, info = large_env.reset(options={"circuit_idx": val_circuit_idx})
print(f"Manually selected circuit: {info['circuit_idx']} (validation)")

### 9.3. Customizing the Reward Function

In [ ]:
# Different reward configurations for different learning behaviors

# 1. Linear reward (simple, but can be unstable)
linear_reward = RewardConfig(
    metric="trace",
    function="linear",
    alpha=10.0,
)

# 2. Log reward (smoother gradients)
log_reward = RewardConfig(
    metric="fidelity",
    function="log",
    alpha=100.0,
)

# 3. Inverted reward (emphasizes small improvements)
inverted_reward = RewardConfig(
    metric="trace",
    function="inverted",
    alpha=50.0,
)

# Compare rewards for different distances
test_distances = [0.001, 0.01, 0.05, 0.1, 0.2]

reward_configs_to_test = [
    ("Linear", linear_reward),
    ("Log", log_reward),
    ("Inverted", inverted_reward),
    ("Inverted Squared", reward_config),  # Our current one
]

print("Reward comparison for different distances:")
print("\nDistance | " + " | ".join([name for name, _ in reward_configs_to_test]))
print("-" * 80)

for dist in test_distances:
    rewards = []
    for name, cfg in reward_configs_to_test:
        rf = RewardFunction(cfg)
        # Create dummy density matrices with controlled distance
        dm1 = np.eye(2, dtype=complex) * 0.5
        dm2 = dm1 + np.eye(2, dtype=complex) * dist
        dm2 = dm2 / np.trace(dm2)  # Normalize
        r = rf(dm1, dm2, is_terminal=True)
        rewards.append(f"{r:8.2f}")
    
    print(f"{dist:8.3f} | " + " | ".join(rewards))

print("\n💡 Choose based on your needs:")
print("  - Linear: Simple, but large rewards might cause instability")
print("  - Log: Smoother, better for stable learning")
print("  - Inverted: Emphasizes getting close to zero distance")
print("  - Inverted Squared: Even more emphasis on precision")

### 9.4. Multi-Qubit Environments

In [ ]:
# Create a 2-qubit environment
multiqubit_config = DatasetConfig(
    n_circuits=20,
    qubits=2,                          # 2 qubits!
    moments=8,
    primitive_gates=["rx", "rz", "cz"],  # Include 2-qubit gate
    clifford=True,
    seed=42
)

multiqubit_noise = NoiseConfig(
    noise_list=[
        GateSpecificNoise(gate="rx", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="rz", noise_channel="depolarizing", noise_parameter=0.03),
        GateSpecificNoise(gate="cz", noise_channel="depolarizing", noise_parameter=0.05),
    ]
)

multiqubit_generator = DatasetGenerator(multiqubit_config, multiqubit_noise)
multiqubit_dataset = multiqubit_generator.generate(verbose=False)

# Create 2-qubit encoder and environment
multiqubit_encoder = CircuitEncoder(
    primitive_gates=["rx", "rz", "cz"],
    qubits=2
)

multiqubit_env = QuantumCircuitEnv(
    dataset=multiqubit_dataset,
    encoder=multiqubit_encoder,
    env_config=env_config,
    reward_config=reward_config,
    primitive_gates=["rx", "rz", "cz"],
)

print("Multi-qubit environment created!")
print(f"Observation space: {multiqubit_env.observation_space.shape}")
print(f"  - Now shows {multiqubit_env.observation_space.shape[1]} qubits")
print(f"\nAction space: {multiqubit_env.action_space.shape}")
print(f"  - Must provide actions for {multiqubit_env.action_space.shape[0]} qubits")
print(f"  - Total action dimension: {np.prod(multiqubit_env.action_space.shape)}")

# Test it
obs, info = multiqubit_env.reset()
print(f"\nObservation shape: {obs.shape}")
print("  (encoding_dim, n_qubits, kernel_size)")

# Action must now have shape (2, 4) for 2 qubits
action = multiqubit_env.action_space.sample()
print(f"\nAction shape: {action.shape}")
print("  (n_qubits, 4 noise parameters)")
print("\nAction values:")
print(action)

### 9.5. Restricting to Only Depolarizing Noise

### 9.6. Only Depolarizing Noise (Simplified Learning)

In [ ]:
# Sometimes you only care about depolarizing noise
# This simplifies the learning problem

depol_only_config = GymEnvConfig(
    kernel_size=3,
    action_space_max_value=0.1,
    val_split=0.2,
    enable_only_depolarizing=True,    # Only depolarizing!
)

depol_env = QuantumCircuitEnv(
    dataset=large_dataset,
    encoder=encoder,
    env_config=depol_only_config,
    reward_config=reward_config,
    primitive_gates=["rx", "rz"],
)

print("Depolarizing-only environment created!")
print(f"Action space still has shape {depol_env.action_space.shape}")
print("But only index 3 (depolarizing) will be used.")
print("Indices 0-2 (epsilon_x, epsilon_z, damping) are automatically zeroed.")

### 9.7. Different Action Strategies

In [ ]:
# Beyond random actions, you can implement various strategies

# 1. Constant noise (baseline)
def constant_action(env, value=0.5):
    """Apply constant noise value."""
    action = np.ones(env.action_space.shape, dtype=np.float32) * value
    return action

# 2. Position-dependent noise
def position_dependent_action(env, position):
    """Increase noise based on circuit depth."""
    action = np.ones(env.action_space.shape, dtype=np.float32)
    # More noise at later positions
    scale = position / env.circuit_length
    return action * scale

# 3. Gate-type aware (would need to parse observation)
def smart_action(obs):
    """Analyze observation to decide action."""
    action = np.zeros((obs.shape[1], 4), dtype=np.float32)
    # Example: higher noise if more gates in window
    gate_density = np.mean(obs[0, :, :])  # Average gate presence
    action[:, 3] = gate_density  # Apply proportional depolarizing
    return action

# Test these strategies
print("Testing action strategies on one episode each:\n")

for strategy_name, strategy_func in [
    ("Random", lambda o, p, e: e.action_space.sample()),
    ("Constant 0.3", lambda o, p, e: constant_action(e, 0.3)),
    ("Position-dependent", lambda o, p, e: position_dependent_action(e, p)),
    ("Smart (gate-aware)", lambda o, p, e: smart_action(o)),
]:
    env.reset()
    terminated = False
    pos = 0
    while not terminated:
        obs = env._get_observation()
        action = strategy_func(obs, pos, env)
        _, reward, terminated, _, _ = env.step(action)
        pos += 1
    
    print(f"{strategy_name:20s}: Final reward = {reward:8.2f}")

### 9.8. Adjusting the Window Size

In [ ]:
# The kernel_size controls how much context the agent sees

# Small window (kernel_size=1): Agent only sees current position
small_window_config = GymEnvConfig(kernel_size=1, action_space_max_value=0.1, val_split=0.0)

# Large window (kernel_size=5): Agent sees more context
large_window_config = GymEnvConfig(kernel_size=5, action_space_max_value=0.1, val_split=0.0)

# Create both environments
small_env = QuantumCircuitEnv(dataset, encoder, small_window_config, reward_config, ["rx", "rz"])
large_env = QuantumCircuitEnv(dataset, encoder, large_window_config, reward_config, ["rx", "rz"])

print("Window size comparison:")
print(f"\nSmall window (kernel_size=1):")
print(f"  Observation shape: {small_env.observation_space.shape}")
print(f"  Agent sees only the current moment")

print(f"\nLarge window (kernel_size=5):")
print(f"  Observation shape: {large_env.observation_space.shape}")
print(f"  Agent sees 2 moments before and 2 moments after")

print("\n💡 Trade-off:")
print("  - Larger window: More context, but more complex observation")
print("  - Smaller window: Simpler, but less information")
print("  - Must be odd number (so there's always a 'current' position)")

## 10. Summary and Next Steps

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           Quantum Circuit Environment Summary                ║
╚══════════════════════════════════════════════════════════════╝

🎯 What You Learned:

1. Dataset Creation
   - Generate circuits with noise
   - Understand circuit encoding (gates + noise parameters)
   - Create datasets with one or multiple circuits

2. Observations (Sliding Window)
   - Agent sees a kernel_size window of the circuit
   - Observation shape: (encoding_dim, n_qubits, kernel_size)
   - Moves forward one step at a time

3. Actions (Noise Parameters)
   - Action shape: (n_qubits, 4)
   - 4 parameters: [epsilon_x, epsilon_z, damping, depolarizing]
   - Scaled by action_space_max_value
   - Applied at current position (middle of window)

4. Rewards (Quality Metric)
   - Only computed at end of episode
   - Metric: How different are the density matrices?
   - Transform: Convert distance to reward (higher is better)
   - Customizable with different metrics and functions

5. Customization Options
   - Multi-qubit circuits
   - Different reward functions
   - Only depolarizing noise (simpler)
   - Adjust window size
   - Train/validation splits

📚 Next Steps:

1. Train an RL agent (PPO, SAC, etc.) with this environment
2. Evaluate on validation circuits
3. Analyze learned noise models
4. Apply to real quantum hardware
5. Experiment with different configurations

💡 Key Files:
   - gym_env.py: Environment implementation
   - reward.py: Reward function options
   - config.py: Configuration models
   - dataset.py: Dataset generation

🔗 Environment follows OpenAI Gym/Gymnasium API:
   - reset() → (observation, info)
   - step(action) → (observation, reward, terminated, truncated, info)
   - Compatible with stable-baselines3, RLlib, etc.

Happy learning! 🚀
""")

## Appendix: Quick Reference

### Configuration Options

**DatasetConfig:**
- `n_circuits`: Number of circuits to generate
- `qubits`: Number of qubits
- `moments`: Circuit depth (number of time steps)
- `primitive_gates`: List of allowed gates (e.g., ["rx", "rz", "cz"])
- `clifford`: Whether to generate Clifford circuits
- `seed`: Random seed for reproducibility

**NoiseConfig:**
- `noise_list`: List of `GateSpecificNoise` objects
  - `gate`: Gate name (e.g., "rx")
  - `noise_channel`: Type ("depolarizing", "damping", "coherent_x", "coherent_z")
  - `noise_parameter`: Noise strength (float or list per qubit)
  - `angle_dependent`: Scale by gate angle (for coherent errors)

**GymEnvConfig:**
- `kernel_size`: Sliding window size (must be odd, default: 3)
- `action_space_max_value`: Max noise value (default: 0.1)
- `val_split`: Validation fraction (default: 0.2)
- `enable_only_depolarizing`: Restrict to only depolarizing (default: False)

**RewardConfig:**
- `metric`: Distance metric ("trace", "fidelity", "mse", "mae")
- `function`: Transform ("linear", "log", "inverted", "inverted_squared")
- `alpha`: Scaling parameter (default: 20.0)

### Space Shapes

- **Observation**: `(encoding_dim, n_qubits, kernel_size)`
  - encoding_dim = 8 (gate info + 4 noise parameters)
  
- **Action**: `(n_qubits, 4)`
  - [0] = epsilon_x
  - [1] = epsilon_z
  - [2] = damping
  - [3] = depolarizing

### Common Patterns

```python
# Create environment from scratch
dataset = DatasetGenerator(dataset_config, noise_config).generate()
encoder = CircuitEncoder(primitive_gates, qubits)
env = QuantumCircuitEnv(dataset, encoder, env_config, reward_config, primitive_gates)

# Run episode
obs, info = env.reset()
terminated = False
while not terminated:
    action = policy(obs)  # Your policy here
    obs, reward, terminated, truncated, info = env.step(action)
```